# Muhtemel Ask - 01 PREPARE TR (V2)

Run all cells in a Google Colab GPU runtime. Change only `EPISODE` and `SOURCE_URL` for normal use. This V2 notebook creates a Turkish-correction pack before forced alignment or subtitle segmentation.

In [ ]:
EPISODE = 12  # @param {type:"integer"}
SOURCE_URL = "https://youtu.be/yVCzw_dFZA8?si=Uq6X2sYPvSrY7Tcr"  # @param {type:"string"}

if EPISODE < 1:
    raise ValueError("EPISODE must be a positive integer")
if not SOURCE_URL.startswith(("https://www.youtube.com/", "https://youtu.be/", "https://www.dailymotion.com/", "https://dai.ly/")):
    raise ValueError("Set SOURCE_URL to the permitted YouTube or Dailymotion source")

In [ ]:
# Advanced settings - normal use does not require changes.
WHISPER_MODEL = "large-v3"  # @param {type:"string"}
BATCH_SIZE = 250  # @param {type:"integer"}
FORCE_REBUILD = False  # @param {type:"boolean"}
EXTRA_AUDIO_REVIEW_UIDS = []  # @param {type:"raw"}

if not 1 <= BATCH_SIZE <= 500:
    raise ValueError("BATCH_SIZE must be between 1 and 500")
if (
    not isinstance(EXTRA_AUDIO_REVIEW_UIDS, list)
    or any(not isinstance(uid, str) or not uid.strip() for uid in EXTRA_AUDIO_REVIEW_UIDS)
    or len(EXTRA_AUDIO_REVIEW_UIDS) != len(set(EXTRA_AUDIO_REVIEW_UIDS))
):
    raise ValueError("EXTRA_AUDIO_REVIEW_UIDS must be a unique list of exact utterance UIDs")

## Mount Drive and install the V2 Colab runtime dependencies

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
import importlib
import os
import shutil
import subprocess
import sys

SYSTEM_ROOT = Path("/content/drive/MyDrive/Muhtemel_Ask_Subtitles/SYSTEM_V2_BETA")
if not (SYSTEM_ROOT / "src").is_dir():
    raise FileNotFoundError(
        f"V2 beta system files were not found at {SYSTEM_ROOT}. Follow V2_BETA_README.md."
    )
if shutil.which("ffmpeg") is None or shutil.which("ffprobe") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=True)
if shutil.which("deno") is None:
    deno_installer = Path("/tmp/deno-install.sh")
    subprocess.run(
        ["curl", "-fsSL", "https://deno.land/install.sh", "-o", str(deno_installer)],
        check=True,
    )
    deno_env = dict(os.environ)
    deno_env["DENO_INSTALL"] = "/usr/local"
    subprocess.run(["sh", str(deno_installer)], check=True, env=deno_env)
if shutil.which("deno") is None:
    raise RuntimeError("Deno installation failed; yt-dlp requires a supported JS runtime")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     str(SYSTEM_ROOT / "requirements-colab.txt")],
    check=True,
)
if str(SYSTEM_ROOT) not in sys.path:
    sys.path.insert(0, str(SYSTEM_ROOT))
for module_name in tuple(sys.modules):
    if module_name == "src" or module_name.startswith("src."):
        del sys.modules[module_name]
importlib.invalidate_caches()
print("V2 beta runtime ready.")

## Create the isolated episode workspace

In [ ]:
import yaml

with (SYSTEM_ROOT / "config/series.yaml").open(encoding="utf-8") as f:
    series_config = yaml.safe_load(f)
with (SYSTEM_ROOT / "config/names.yaml").open(encoding="utf-8") as f:
    names_config = yaml.safe_load(f)
with (SYSTEM_ROOT / "config/religious_terms.yaml").open(encoding="utf-8") as f:
    religious_config = yaml.safe_load(f)

EPISODE_NAME = f"Muhtemel Ask {EPISODE}.Bolum"
EPISODE_ROOT = Path(series_config["drive_root"]) / "EPISODES" / EPISODE_NAME
DIRS = {name: EPISODE_ROOT / name for name in (
    "source", "prepare", "translation_input",
    "translation_output", "review", "final"
)}
for folder in DIRS.values():
    folder.mkdir(parents=True, exist_ok=True)
print(f"Episode workspace: {EPISODE_ROOT}")

## Reusable YouTube authentication

In [ ]:
YOUTUBE_COOKIE_DRIVE_PATH = (
    Path(series_config["drive_root"]) / "PRIVATE" / "youtube-cookies.txt"
)
YOUTUBE_COOKIE_RUNTIME_ROOT = Path("/content")


def normalise_and_validate_youtube_cookie_payload(payload):
    """Return safe Netscape cookie bytes containing YouTube domains only."""
    import http.cookiejar
    import tempfile
    import warnings

    payload = bytes(payload)
    if not 0 < len(payload) <= 5 * 1024 * 1024:
        raise ValueError("The cookie file must be between 1 byte and 5 MiB")
    payload = payload.removeprefix(b"\xef\xbb\xbf")
    payload = payload.replace(b"\r\n", b"\n").replace(b"\r", b"\n")
    first_line = payload.split(b"\n", 1)[0]
    if first_line not in {b"# HTTP Cookie File", b"# Netscape HTTP Cookie File"}:
        raise ValueError("Use a Netscape-format cookies.txt file")
    check_dir = Path(tempfile.mkdtemp(
        prefix=".muhtemel-ask-cookie-check-", dir=str(YOUTUBE_COOKIE_RUNTIME_ROOT)
    ))
    os.chmod(check_dir, 0o700)
    check_path = check_dir / "cookies.txt"
    try:
        check_path.write_bytes(payload)
        os.chmod(check_path, 0o600)
        jar = http.cookiejar.MozillaCookieJar(str(check_path))
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", UserWarning)
                jar.load(ignore_discard=True, ignore_expires=True)
        except (OSError, UnicodeError, http.cookiejar.LoadError):
            raise ValueError("The cookies.txt file is invalid") from None
        cookies = list(jar)
        if not cookies:
            raise ValueError("The cookies.txt file contains no cookies")
        domains = {cookie.domain.lstrip(".").casefold() for cookie in cookies}
        if not all(domain == "youtube.com" or domain.endswith(".youtube.com") for domain in domains):
            raise ValueError("Export only youtube.com cookies, not all browser cookies")
        return payload
    finally:
        shutil.rmtree(check_dir, ignore_errors=True)


def stage_youtube_cookies(payload):
    """Create the only cookie copy that yt-dlp is allowed to read."""
    import tempfile

    payload = normalise_and_validate_youtube_cookie_payload(payload)
    auth_dir = Path(tempfile.mkdtemp(
        prefix=".muhtemel-ask-youtube-", dir=str(YOUTUBE_COOKIE_RUNTIME_ROOT)
    ))
    os.chmod(auth_dir, 0o700)
    target = auth_dir / "cookies.txt"
    try:
        target.write_bytes(payload)
        os.chmod(target, 0o600)
        return target
    except BaseException:
        shutil.rmtree(auth_dir, ignore_errors=True)
        raise


def load_saved_youtube_cookies():
    """Copy the saved Drive credential into the private runtime."""
    if not YOUTUBE_COOKIE_DRIVE_PATH.is_file():
        return None
    try:
        cookie_path = stage_youtube_cookies(YOUTUBE_COOKIE_DRIVE_PATH.read_bytes())
    except (OSError, ValueError):
        print("Saved YouTube cookies are unreadable or invalid; upload a fresh file.")
        return None
    print("Saved YouTube cookies loaded from Drive.")
    return cookie_path


def save_youtube_cookies_to_drive(cookie_path):
    """Atomically replace the reusable Drive copy after validation."""
    import tempfile

    try:
        payload = normalise_and_validate_youtube_cookie_payload(Path(cookie_path).read_bytes())
        YOUTUBE_COOKIE_DRIVE_PATH.parent.mkdir(parents=True, exist_ok=True)
        descriptor, temporary_name = tempfile.mkstemp(
            prefix=".youtube-cookies.", suffix=".tmp",
            dir=str(YOUTUBE_COOKIE_DRIVE_PATH.parent),
        )
        temporary_path = Path(temporary_name)
        try:
            with os.fdopen(descriptor, "wb") as handle:
                handle.write(payload)
                handle.flush()
            os.replace(temporary_path, YOUTUBE_COOKIE_DRIVE_PATH)
        except BaseException:
            temporary_path.unlink(missing_ok=True)
            raise
    except (OSError, ValueError):
        print("WARNING: YouTube cookies could not be saved to Drive.")
        return False
    print("YouTube cookies saved to Drive for later runs.")
    return True


def upload_youtube_cookies():
    """Upload one YouTube-only Netscape cookie file into this runtime."""
    from google.colab import files
    import tempfile

    upload_dir = Path(tempfile.mkdtemp(
        prefix=".muhtemel-ask-upload-", dir=str(YOUTUBE_COOKIE_RUNTIME_ROOT)
    ))
    os.chmod(upload_dir, 0o700)
    previous_cwd = Path.cwd()
    uploaded = {}
    try:
        os.chdir(upload_dir)
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError("Upload exactly one YouTube cookies.txt file")
        cookie_path = stage_youtube_cookies(next(iter(uploaded.values())))
        print("YouTube cookies loaded into the temporary Colab runtime.")
        return cookie_path
    finally:
        os.chdir(previous_cwd)
        uploaded.clear()
        shutil.rmtree(upload_dir, ignore_errors=True)


def discard_youtube_cookies(cookie_path):
    if cookie_path is None:
        return
    auth_dir = Path(cookie_path).resolve().parent
    runtime_root = YOUTUBE_COOKIE_RUNTIME_ROOT.resolve()
    if auth_dir.parent != runtime_root or not auth_dir.name.startswith(".muhtemel-ask-youtube-"):
        raise RuntimeError("Refusing to remove unexpected cookie directory")
    shutil.rmtree(auth_dir, ignore_errors=True)

## Download, probe and extract audio

In [ ]:
from src.download import YouTubeAuthenticationError, download_source
from src.media import extract_audio, save_source_metadata

try:
    download = download_source(
        SOURCE_URL, DIRS["source"], language="tr", force=FORCE_REBUILD,
        output_stem=EPISODE_NAME,
    )
except YouTubeAuthenticationError:
    print("YouTube requires browser verification; checking the saved Drive credential.")
    youtube_cookie_file = load_saved_youtube_cookies()
    if youtube_cookie_file is not None:
        try:
            download = download_source(
                SOURCE_URL, DIRS["source"], language="tr", force=FORCE_REBUILD,
                output_stem=EPISODE_NAME, cookies_file=youtube_cookie_file,
            )
        except YouTubeAuthenticationError:
            download = None
            print("Saved YouTube cookies were rejected; upload a fresh file.")
        finally:
            discard_youtube_cookies(youtube_cookie_file)
    else:
        download = None
    if download is None:
        youtube_cookie_file = upload_youtube_cookies()
        try:
            download = download_source(
                SOURCE_URL, DIRS["source"], language="tr", force=FORCE_REBUILD,
                output_stem=EPISODE_NAME, cookies_file=youtube_cookie_file,
            )
            save_youtube_cookies_to_drive(youtube_cookie_file)
        finally:
            discard_youtube_cookies(youtube_cookie_file)
source_metadata = save_source_metadata(
    download.video_path,
    DIRS["source"] / "source.media.json",
    original_url=SOURCE_URL,
    download_metadata=download.metadata,
)
audio = extract_audio(
    download.video_path,
    DIRS["prepare"],
    source_sha256=source_metadata["sha256"],
    force=FORCE_REBUILD,
)
print("Download resumed:", download.resumed)
print("Audio resumed:", audio.resumed)

## Coarse Turkish ASR, independent speech coverage and rescue

The timings produced here are provisional evidence only. Final subtitle timing is created later from corrected Turkish using forced alignment.

In [ ]:
from src.raw_asr_v2 import RawASRV2Config, transcribe_raw_audio_v2

raw_asr = transcribe_raw_audio_v2(
    audio.audio_path,
    DIRS["prepare"],
    episode=EPISODE,
    config=RawASRV2Config(
        model_name=WHISPER_MODEL,
        extra_audio_review_uids=tuple(EXTRA_AUDIO_REVIEW_UIDS),
    ),
    captions_path=download.captions_path,
    canonical_names=tuple(names_config["canonical_names"]),
    religious_terms=tuple(item["source"] for item in religious_config["terms"]),
    force=FORCE_REBUILD,
)
if raw_asr.get("independent_vad") is not True:
    raise RuntimeError("V2 PREPARE requires independent Silero VAD evidence")
correction_utterances = raw_asr["correction_utterances"]
speech_hole_records = raw_asr["speech_hole_records"]
asr_hallucination_records = raw_asr["asr_hallucination_records"]
unresolved_speech_candidate_count = len(speech_hole_records)
hallucination_review_count = len(asr_hallucination_records)
suspected_asr_count = sum(
    "suspected_asr_hallucination" in item["risk_flags"]
    for item in asr_hallucination_records
)
orphan_caption_count = sum(
    "orphan_youtube_caption" in item["risk_flags"]
    for item in asr_hallucination_records
)
blank_audio_review_count = sum(
    not item["asr_text"].strip()
    and "unresolved_vad_speech" in item["risk_flags"]
    for item in correction_utterances
)
flagged_candidate_count = sum(
    bool({"suspected_asr_hallucination", "orphan_youtube_caption"}.intersection(item["risk_flags"]))
    for item in correction_utterances
)
if blank_audio_review_count != unresolved_speech_candidate_count:
    raise RuntimeError(
        "Every unresolved speech candidate must have one blank correction record"
    )
if flagged_candidate_count != hallucination_review_count:
    raise RuntimeError(
        "Every flagged ASR/caption candidate must have one immutable WAV review record"
    )
print("Raw ASR resumed:", raw_asr["resumed"])
print(f"Correction records: {len(correction_utterances):,}")
print(f"Unresolved speech candidates with WAV evidence: {unresolved_speech_candidate_count:,}")
print(f"Suspected ASR hallucinations with WAV evidence: {suspected_asr_count:,}")
print(f"Orphan YouTube captions with WAV evidence: {orphan_caption_count:,}")
if unresolved_speech_candidate_count or hallucination_review_count:
    print(
        "IMPORTANT: Pro must not open or transcribe the bundled WAVs. "
        "Keep every flagged audio decision pending; notebook 02 reviews only these short clips in Colab."
    )

## Create the Turkish-correction pack

Unresolved speech candidates and ASR/caption candidates keep immutable, hash-bound contextual WAV evidence in this pack. The ChatGPT pass is text-only: it must not open, listen to or transcribe those WAVs. Every linked record stays pending until `02_ALIGN_PREPARE_ID.ipynb` performs the bounded, resumable Colab audio audit.

Every ordinary non-flagged ASR record remains dialogue and receives non-empty corrected Turkish. Every linked review record remains `non_dialogue=false`, `review_required=true`, `audio_reviewed=false`, and `review_disposition=pending_audio_review` in the provisional `_TR_TEXT_CORRECTED.zip`. A blank speech hole keeps empty `tr_corrected`; a text-bearing ASR/caption candidate may have its Turkish text corrected. Colab, not the text model, creates the final `_TR_CORRECTED.zip`.

In [ ]:
from src.tr_correction import (
    create_tr_correction_pack,
    validate_tr_correction_output,
)

pack_path = (
    DIRS["translation_input"]
    / f"{EPISODE_NAME}_TR_CORRECTION_PACK.zip"
)
text_output_path = (
    DIRS["translation_output"]
    / f"{EPISODE_NAME}_TR_TEXT_CORRECTED.zip"
)
had_text_output = text_output_path.is_file()
manifest = create_tr_correction_pack(
    correction_utterances,
    speech_hole_records,
    pack_path,
    episode=EPISODE,
    batch_size=BATCH_SIZE,
    speech_hole_audio_root=DIRS["prepare"],
    asr_hallucination_records=asr_hallucination_records,
    asr_hallucination_audio_root=DIRS["prepare"],
    rebind_text_output_path=text_output_path,
)
expected_name = f"{EPISODE_NAME}_TR_CORRECTION_PACK.zip"
if pack_path.name != expected_name or not pack_path.is_file():
    raise RuntimeError("TR correction pack was not published at the exact path")
if manifest["utterance_count"] != len(correction_utterances):
    raise RuntimeError("TR correction pack utterance count mismatch")
if manifest["speech_hole_count"] != unresolved_speech_candidate_count:
    raise RuntimeError("TR correction pack speech-hole count mismatch")
if manifest["asr_hallucination_count"] != hallucination_review_count:
    raise RuntimeError("TR correction pack hallucination-review count mismatch")
if had_text_output:
    rebound = validate_tr_correction_output(pack_path, text_output_path)
    print(
        f"Existing text correction safely rebound: {len(rebound.records):,} records; "
        "Pro does not need to repeat the Turkish correction."
    )
print(f"TR correction input SHA-256: {manifest['input_sha256']}")
print(f"Speech-hole clips reserved for Colab review: {unresolved_speech_candidate_count:,}")
print(f"ASR/caption clips reserved for Colab review: {hallucination_review_count:,}")
print(f"TR text-correction pack ready: {pack_path}")